# Forbes America's Top Colleges — Exploratory Data Analysis

### A Guided Practice Notebook (Questions Only)

This notebook walks through a real-world exploratory data analysis (EDA) exercise using the
**Forbes America's Top Colleges** dataset. Each section below poses a question, explains the
underlying pandas concept, gives you a hint, and describes what your output should look like.
Your task is to write the code yourself in the empty cell provided under each question.

---

## Context

Since 2008, Forbes Magazine has published an annual ranking of America's best colleges. For a
question as important as *"Is college worth it?"*, this list is a widely cited reference point —
helping prospective students weigh academics, campus experience, career outcomes, and student debt
before choosing where to study.

The Forbes ranking methodology reviews undergraduate institutions across several dimensions:
academic quality, student experience, post-graduation career success, and affordability
(student debt levels).

## About the Data

The dataset contains rankings and statistics for **650 United States colleges**, including fields
such as rank, name, location (state/city), public or private status, student population,
undergraduate population, acceptance rate, average grant aid, and average alumni salary.

## Learning Objectives

By the end of this notebook, you should be comfortable with:

- Loading and inspecting a dataset with pandas (`shape`, `info`, `describe`)
- Detecting and handling missing values
- Cleaning up column names and resetting indexes
- Filtering rows with boolean conditions
- Grouping and aggregating data with `groupby`
- Sorting values to find "top N" style answers
- Creating new calculated columns
- Searching text columns with string methods
- Binning a continuous variable into categories with `pd.cut`
- Selecting data by position and label using `iloc` and `loc`

---


## Setup

### Load Required Packages

We will use **pandas** for data manipulation and **numpy** for numerical operations. These are the
two core libraries for almost any data analysis task in Python.


In [ ]:
import os
import pandas as pd
import numpy as np

os.getcwd()


### Load the Dataset

Make sure the file `AmericasTopColleges.csv` is saved in the same folder as this notebook
(or update the path below) before running the cell.


In [ ]:
ac = pd.read_csv("AmericasTopColleges.csv")


---

## Warm-Up Exercises

Before jumping into analysis questions, it's good practice to always inspect a new dataset first:
how big is it, what type of data does each column hold, and is anything missing or messy? The
exercises below build that habit.


### Warm-Up 1: Check the Dimensions of the Data

**Concept:** The `.shape` attribute of a DataFrame returns a tuple `(number_of_rows, number_of_columns)`.
It's usually the very first thing you check after loading any dataset, so you know the scale of
what you're working with.

**Hint:** Simply call the `.shape` attribute on your DataFrame — no parentheses needed, since it's
an attribute, not a method.

**Expected Output:** A tuple of two integers, e.g. `(650, N)`, representing rows and columns.


In [ ]:
ac.shape

### Warm-Up 2: Check the Data Types of Each Field

**Concept:** The `.info()` method summarizes the DataFrame: column names, non-null counts, and data
types (`int64`, `float64`, `object`, etc.). This tells you which columns are numeric and which are
text — important before you try to do math on a column.

**Hint:** Call `.info()` on your DataFrame.

**Expected Output:** A printed summary listing every column name, its non-null count, and its dtype,
plus total memory usage.


In [ ]:
ac.info()

### Warm-Up 3: Check and Handle Missing Values

**Concept:** `.isnull()` returns a same-shaped DataFrame of `True`/`False` values marking where data
is missing. Chaining `.sum()` after it adds up the `True` values per column, giving you a missing-value
count per column. Once you know where the gaps are, a common cleanup approach is to fill missing
**text** columns with the most frequent value (`mode`) and missing **numeric** columns with the
`median` (median is preferred over mean here since it's less sensitive to outliers).

**Hint:** First run `ac.isnull().sum()` to see the gaps. Then loop through each column, checking
`ac[column].dtype`, and use `.fillna()` with `.mode()[0]` for object/text columns or `.median()` for
numeric columns. Remember to pass `inplace=True` (or reassign the column) so the change is saved.

**Expected Output:** First, a Series showing missing-value counts per column. After cleanup, running
`ac.isnull().sum()` again should show all zeros.


In [ ]:
# Check missing values per column
print("Missing values before cleanup:")
print(ac.isnull().sum())

# Handle missing values: mode for object/text, median for numeric
for column in ac.columns:
    if ac[column].dtype == 'object' or ac[column].dtype == 'str':
        ac[column] = ac[column].fillna(ac[column].mode()[0])
    else:
        ac[column] = ac[column].fillna(ac[column].median())

# Check missing values after cleanup
print("\nMissing values after cleanup:")
print(ac.isnull().sum())

### Warm-Up 4: Preview the First and Last Rows

**Concept:** `.head()` and `.tail()` show the first/last 5 rows of a DataFrame by default (you can
pass a number, e.g. `.head(10)`, for more). This is a quick sanity check to visually confirm the data
loaded correctly and looks as expected.

**Hint:** Call `.head()` and `.tail()` separately on your DataFrame.

**Expected Output:** Two small tables, each with 5 rows and all columns, showing the beginning and end
of the dataset.


In [ ]:
# Preview first 5 rows
display(ac.head())

# Preview last 5 rows
display(ac.tail())

### Warm-Up 5: Clean Up Column Names

**Concept:** Column names with spaces or special characters (like `/`) are inconvenient to reference
in code, since `ac.Average Grant Aid` would break — you'd need `ac["Average Grant Aid"]` instead.
Standardizing column names (e.g., replacing spaces and slashes with underscores) lets you use the
cleaner dot notation (`ac.Average_Grant_Aid`) throughout the rest of the analysis.

**Hint:** Use `ac.columns.str.replace()` with a pattern matching spaces and slashes, replacing them
with an underscore, and reassign the result back to `ac.columns`.

**Expected Output:** Running `ac.columns` afterward should show column names with underscores instead
of spaces or slashes (e.g., `Public_Private` instead of `Public/Private`).


In [ ]:
# Standardize column names: replace spaces and slashes with underscores
ac.columns = ac.columns.str.replace(r'[ /]', '_', regex=True)
ac.columns

### Warm-Up 6: Get a Statistical Summary

**Concept:** `.describe()` computes summary statistics (count, mean, standard deviation, min, max,
and quartiles) for all numeric columns in one call — a fast way to spot outliers or unusual ranges
before deeper analysis.

**Hint:** Call `.describe()` on your DataFrame.

**Expected Output:** A table with statistics (count, mean, std, min, 25%, 50%, 75%, max) as rows, and
each numeric column as a column.


In [ ]:
ac.describe()

---

## Analysis Questions

Now that the data is clean and you're familiar with its structure, work through the following
business-style questions. Each one mirrors the kind of question a real analyst might be asked about
this dataset.


### Question 1: Which Are the Top 5 Colleges in the Forbes Ranking?

**Business Context:** Before any deeper analysis, stakeholders usually want a quick answer:
who's at the very top?

**Concept:** This is **boolean/conditional filtering** — selecting rows of a DataFrame where a
condition on a column evaluates to `True`. Here the condition is based on the `Rank` column.

**Hint:** Try a condition like `ac.Rank <= 5` inside square brackets, or sort by `Rank` and use
`.head()`.

**Expected Output:** A DataFrame with 5 rows (all columns), showing the colleges ranked 1 through 5.


In [ ]:
# Colleges ranked 1 through 5
ac[ac.Rank <= 5]

### Question 2: Which State and City Do the Top 5 Colleges Belong To?

**Business Context:** Knowing location alongside rank helps identify regional clusters of
top-performing institutions.

**Concept:** **Column selection** — once you have a filtered/sliced set of rows, you can select
only specific columns of interest by passing a list of column names in double square brackets.

**Hint:** Combine your Question 1 approach with selecting only the `State`, `City`, and `Rank`
columns, e.g. `df[["State", "City", "Rank"]]`.

**Expected Output:** A DataFrame with 5 rows and exactly 3 columns: `State`, `City`, `Rank`.


In [ ]:
# State, City, and Rank of the top 5 colleges
ac[ac.Rank <= 5][["State", "City", "Rank"]]

### Question 3: How Many States (and Which Ones) Have Colleges Listed by Forbes?

**Business Context:** Understanding geographic spread tells us whether this ranking favors certain
regions of the country.

**Concept:** `.nunique()` counts the number of **unique** values in a column. `.groupby()` combined
with `.count()` groups rows by a category (here, `State`) and counts how many rows fall into each
group — useful for "how many per category" questions.

**Hint:** Use `ac.State.nunique()` for the count of unique states, and
`ac.groupby(by="State").count()` (selecting one column, e.g. `["Name"]`, to keep the output clean)
to see how many colleges per state.

**Expected Output:** A single integer for the unique state count, and a Series/table listing each
state alongside its number of colleges.


In [ ]:
# Count of unique states
print("Unique states count:", ac.State.nunique())

# Number of colleges per state
ac.groupby(by="State")["Name"].count()

### Question 4: How Many Private vs. Public Colleges Are in the List?

**Business Context:** Public vs. private status often correlates with tuition cost and funding
source — a useful segmentation for prospective students comparing affordability.

**Concept:** `.value_counts()` is a quick way to count how many times each unique value appears in a
column — ideal for categorical columns with a small number of categories (like Public vs. Private).

**Hint:** Make sure you've cleaned the column name first (see Warm-Up 5), then call
`.value_counts()` on the `Public_Private` column.

**Expected Output:** A Series with two entries (`Public` and `Private`) and their respective counts,
which should sum to the total number of rows in the dataset.


In [ ]:
# Count of Public vs. Private colleges
ac.Public_Private.value_counts()

### Question 5: How Many Colleges Are There Within Each State? Which State Has the Most?

**Business Context:** This tells us which states are the most competitive/concentrated markets for
top-ranked colleges.

**Concept:** Building on `.groupby()` and `.count()` from Question 3, you can chain `.sort_values()`
to order the results, and use `.index[0]` to pull out just the top state's name.

**Hint:** Sort your grouped state counts in descending order with `.sort_values(ascending=False)`,
then use `.index[0]` to get the state with the highest count.

**Expected Output:** A sorted Series (state → count of colleges), and separately, a single string
value naming the state with the most colleges.


In [ ]:
# Number of colleges within each state, sorted descending
state_counts = ac.groupby(by="State")["Name"].count().sort_values(ascending=False)
display(state_counts)

# State with the most colleges
print("\nState with the most colleges:", state_counts.index[0])

### Question 6: What Percentage of Students Are Undergraduates at Each College?

**Business Context:** A high undergraduate percentage may indicate a college focused primarily on
undergraduate education rather than graduate/research programs.

**Concept:** This is about **creating a new calculated column**. You can derive a new field from
existing ones using standard arithmetic operations directly on columns — pandas applies the
operation row-by-row automatically (this is called "vectorized" computation).

**Hint:** Divide the undergraduate population column by the total student population column,
multiply by 100, and round the result (e.g., using `.round(2)`). Assign this to a new column name.

**Expected Output:** The DataFrame should now have one additional column (e.g., `UnderGradPerc`)
containing percentage values, typically between 0 and 100.


In [ ]:
# Calculate percentage of undergraduate students
ac["UnderGradPerc"] = ((ac.Undergraduate_Population / ac.Student_Population) * 100).round(2)
ac[["Name", "Undergraduate_Population", "Student_Population", "UnderGradPerc"]].head()

### Question 7: Which 5 Colleges Have the Highest Percentage of Undergraduates?

**Business Context:** This narrows down Question 6 into an actionable "top 5" list.

**Concept:** `.sort_values(by=..., ascending=False)` orders rows by a specific column from highest
to lowest. Combine with `.head()` to get just the top N rows, and column selection to keep only the
relevant fields.

**Hint:** Sort by the new percentage column you created in Question 6, in descending order, then
take the top 5 rows and select just the `Name` and percentage columns.

**Expected Output:** A DataFrame with 5 rows and 2 columns: college name and undergraduate
percentage, sorted from highest to lowest.


In [ ]:
# Top 5 colleges with highest percentage of undergraduates
ac.sort_values(by="UnderGradPerc", ascending=False)[["Name", "UnderGradPerc"]].head(5)

### Question 8: Which Colleges Receive the Highest Average Grant Aid?

**Business Context:** Grant aid (money that doesn't need to be repaid) is a major factor in
affordability — colleges offering the most aid may be more attractive to cost-conscious students.

**Concept:** Same technique as Question 7 — sorting and slicing — just applied to a different
column.

**Hint:** Sort by the `Average_Grant_Aid` column in descending order, then take the top rows and
select the `Name` and `Average_Grant_Aid` columns.

**Expected Output:** A DataFrame listing the top colleges by average grant aid, with college name
and grant aid amount as columns.


In [ ]:
# Colleges receiving highest average grant aid
ac.sort_values(by="Average_Grant_Aid", ascending=False)[["Name", "Average_Grant_Aid"]].head()

### Question 9: What Is the Average Undergraduate Population in the State of California ('CA')?

**Business Context:** Comparing a single state's average against the national picture is a common
"drill-down" style question.

**Concept:** This combines **row filtering** (restrict to one state) with an **aggregate function**
(`.mean()`) applied to a specific column.

**Hint:** Filter the DataFrame where `State` equals `"CA"`, select the undergraduate population
column, and call `.mean()` on it.

**Expected Output:** A single numeric value representing the average undergraduate population across
California colleges in the dataset.


In [ ]:
# Average undergraduate population in California ('CA')
ac[ac.State == "CA"]["Undergraduate_Population"].mean()

### Question 10: Which Colleges Have the Highest Chances of Admission?

**Business Context:** A high acceptance rate generally signals an easier college to get into —
useful for students weighing "reach" vs. "safety" school options.

**Concept:** Boolean filtering again, this time using a threshold condition on the
`Acceptance_Rate` column. Checking `.max()` first helps you understand the range before picking a
sensible threshold.

**Hint:** First check `ac.Acceptance_Rate.max()` to see the highest value in the dataset. Then
filter for colleges above a threshold you choose (e.g., greater than 60), and select the `Name` and
`Acceptance_Rate` columns.

**Expected Output:** A DataFrame listing colleges whose acceptance rate exceeds your chosen
threshold, with their name and acceptance rate.


In [ ]:
# Check highest acceptance rate
print("Maximum Acceptance Rate:", ac.Acceptance_Rate.max())

# Colleges with acceptance rate > 60%
ac[ac.Acceptance_Rate > 60][["Name", "Acceptance_Rate"]]

### Question 11: Are There Any 'Art' Colleges in the List?

**Business Context:** Sometimes you need to search for institutions matching a keyword in their
name, rather than filtering on a numeric threshold.

**Concept:** `.str.contains()` is a **string method** that checks whether a substring appears
anywhere within each value of a text column, returning `True`/`False` per row — which you can then
use as a filter condition.

**Hint:** Use `ac.Name.str.contains("Art")` as your filter condition inside square brackets.

**Expected Output:** A DataFrame containing only the rows where the college name includes the
substring "Art" (this may include colleges where "Art" appears as part of a longer word).


In [ ]:
# Filter colleges with 'Art' in their name
ac[ac.Name.str.contains("Art")][["Rank", "Name"]]

### Question 12: Divide Alumni Salary into Three Buckets (Low, Medium, High)

**Business Context:** Grouping a continuous variable like salary into broad tiers (Low/Medium/High)
makes it easier to summarize and compare across colleges, rather than working with exact figures.

**Concept:** `pd.cut()` **bins** a continuous numeric column into a specified number of equal-width
intervals, and can assign custom labels to each bin — turning a continuous variable into a
categorical one.

**Hint:** Use `pd.cut()` on the `Alumni_Salary` column, specifying `bins=3` and
`labels=["L", "M", "H"]`. Save the result into a new column. Then filter for rows where that new
column equals `"H"`.

**Expected Output:** A new categorical column with values `L`, `M`, or `H` for every row, and a
filtered DataFrame showing only the colleges that fall into the `"H"` (high salary) bucket.


In [ ]:
# Divide Alumni Salary into 3 buckets: Low (L), Medium (M), High (H)
ac["Salary_Bucket"] = pd.cut(ac.Alumni_Salary, bins=3, labels=["L", "M", "H"])

# Filter colleges in the 'H' bucket
ac[ac.Salary_Bucket == "H"]

### Question 13: What Is the Mean Alumni Salary at Each Bucket Level?

**Business Context:** This validates that your bucketing in Question 12 makes sense — the "High"
bucket should indeed have a noticeably higher average salary than "Low."

**Concept:** `.groupby()` combined with `.mean()` computes the average of a numeric column, split
out by each category in another column — here, your salary bucket labels.

**Hint:** Group by your salary bin column and compute `.mean()` on the `Alumni_Salary` column.

**Expected Output:** A small table with 3 rows (one per bucket: L, M, H) showing the average alumni
salary for each group — values should increase from L to H.


In [ ]:
# Mean Alumni Salary at each bucket level
ac.groupby("Salary_Bucket")["Alumni_Salary"].mean()

### Question 14: Create a New DataFrame of Colleges with High Alumni Salary

**Business Context:** Once you've identified a subgroup of interest (colleges in the "High" alumni
salary bucket), it's common practice to save it as its own DataFrame for focused follow-up analysis.

**Concept:** This is simply **filtering and assignment** — the result of a boolean filter can be
stored in a new variable name, creating an independent DataFrame you can analyze further without
disturbing the original data.

**Hint:** Filter `ac` where your salary bucket column equals `"H"`, and assign the result to a new
variable name.

**Expected Output:** A new DataFrame containing only the "High" alumni salary colleges — check its
`.shape` to confirm how many colleges fall into this group.


In [ ]:
# Create DataFrame of colleges with High alumni salary
high_salary_ac = ac[ac.Salary_Bucket == "H"]
print("High Salary DataFrame Shape:", high_salary_ac.shape)
high_salary_ac.head()

---

## Bonus: Indexing with `iloc` and `loc`

**Concept:** Pandas offers two primary ways to select subsets of a DataFrame by position or by
label:
- **`.iloc[]`** selects by **integer position** — e.g., "give me rows 2 and 3" regardless of their
  labels.
- **`.loc[]`** selects by **label** — e.g., "give me rows labeled 1 through 2" and "columns from
  `Public_Private` through `Alumni_Salary`" (note: with `.loc`, the end label is *included*, unlike
  standard Python slicing).

**Hint:** Try `ac.iloc[2:4, :]` to select rows by position, and
`ac.loc[1:2, "Public_Private":"Alumni_Salary"]` to select rows and a column range by label.

**Expected Output:** Two small DataFrames — one showing rows at positions 2 and 3 (all columns), and
one showing rows labeled 1 and 2, restricted to the column range specified.


In [ ]:
# Positional indexing with iloc (rows 2 and 3)
display(ac.iloc[2:4, :])

# Label-based indexing with loc (rows 1-2, columns Public_Private through Alumni_Salary)
display(ac.loc[1:2, "Public_Private":"Alumni_Salary"])

---

## Wrap-Up

You've now practiced the core EDA toolkit in pandas: inspecting structure, cleaning data, filtering,
grouping, sorting, deriving new columns, string search, binning, and positional/label-based
indexing. These same techniques apply directly to almost any tabular dataset you'll encounter —
whether it's college rankings, sales records, or loan applications.

**Happy Learning!**
